# Hands-on 38 (Bab 15): Proyek Akhir End-to-End

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 15.

Capstone: simulator -> MQTT -> subscriber Python (validasi payload, preprocessing,
threshold baseline + Isolation Forest) -> storage -> alert. Struktur skrip mengikuti
buku: `scripts/publishers/`, `scripts/inference/`, `scripts/subscribers/`.

## Tujuan

Pada latihan ini, pembaca akan membangun proyek akhir end-to-end yang utuh, mulai dari:

- data sensor atau simulator,
- pengiriman ke MQTT,
- subscriber Python,
- preprocessing otomatis,
- inferensi anomaly detection,
- penyimpanan hasil,
- hingga dashboard Grafana.

Setelah selesai, pembaca diharapkan mampu menghasilkan sistem mini yang:

- benar-benar hidup,
- dapat dijalankan berulang,
- dan bisa dikembangkan menjadi proyek lapangan yang lebih serius.

Ini adalah praktik paling penting dalam seluruh buku, karena di sinilah semua blok pengetahuan dari bab-bab sebelumnya benar-benar menyatu. Bab ini ditujukan untuk pembaca yang ingin hasil nyata, bukan hanya potongan-potongan teori.

## Skenario proyek yang dipakai

Agar tetap fleksibel, proyek akhir ini dirancang sebagai monitoring dan deteksi anomali perangkat IoT. Data yang dipakai dapat berupa:

- suhu, kelembaban, CO₂,
- arus listrik,
- daya listrik,
- getaran sederhana,
- atau kombinasi sensor yang tersedia.

Untuk menjaga latihan tetap praktis, kita gunakan payload generik berikut:

```json
{
    "timestamp": "2026-01-01T10:10:00",
    "device_id": "iotnode01",
    "temperature_c": 28.6,
    "humidity_pct": 54.1,
    "power_watt": 145.3,
    "current_amp": 5.8
}
```

Ini cukup fleksibel untuk:

- smart room monitoring,
- smart energy node,
- node peralatan kecil,
- atau gateway edge kampus/lab.

## Arsitektur sistem

Arsitektur yang dibangun:

sensor / simulator / synthetic generator → MQTT broker → Python subscriber → preprocessing otomatis → model anomaly detection → storage → dashboard Grafana

Peran tiap komponen

- Publisher: mengirim telemetry sensor
- Broker MQTT: mendistribusikan pesan
- Subscriber Python: validasi, preprocessing, inferensi
- Storage: menyimpan data dan hasil model
- Grafana: menampilkan histori dan status
- Alert: memberi notifikasi saat kondisi abnormal

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /downloads/iot-ml-book


## 1. Latih Model Baseline Isolation Forest

Baseline dilatih pada rentang operasi normal deployment (rentang payload normal
publisher); formula model persis buku. Lihat docstring
`scripts/inference/train_capstone_iforest.py` untuk alasannya.

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/inference/train_capstone_iforest.py"], capture_output=True, text=True)
print(r.stdout)

[SUKSES] Model disimpan ke: data/models/capstone_iforest.joblib



## 2. Jalankan Sistem (broker + subscriber+alert + listener + publisher)

In [3]:
influx_hapus("capstone_device_monitoring"); influx_hapus("capstone_alerts")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = latar("scripts/subscribers/capstone_subscriber_with_alert.py", "capstone_sub_alert_nb.log")
    listener = latar("scripts/capstone_alert_listener.py", "capstone_listener_nb.log", ("--max-messages", "1", "--timeout", "60"))
    time.sleep(2)
    pub = latar("scripts/publishers/capstone_publisher.py", "capstone_pub_nb.log")
    time.sleep(40)   # sekitar 20 pesan (interval 2 detik)
    henti(pub, sub, listener)
    print("--- Subscriber + alert ---")
    print(log_teks("capstone_sub_alert_nb.log"))
    print("--- Listener topik alert ---")
    print(log_teks("capstone_listener_nb.log")[:500])
finally:
    broker.terminate()

--- Subscriber + alert ---
Subscribed to capstone/iotnode01/telemetry
ALERT: {'timestamp': '2026-11-02T12:20:02', 'device_id': 'iotnode01', 'temperature_c': 27.06, 'humidity_pct': 63.72, 'power_watt': 314.14729546288515, 'current_amp': 11.575074842524117, 'reason': 'threshold,iforest'}
ALERT: {'timestamp': '2026-11-02T12:20:24', 'device_id': 'iotnode01', 'temperature_c': 29.82, 'humidity_pct': 69.06, 'power_watt': 262.7897053235855, 'current_amp': 13.903517658748253, 'reason': 'threshold,iforest'}

--- Listener topik alert ---
[INFO] Menunggu alert di topik: capstone/iotnode01/alert

=== NOTIFIKASI ALERT ===
Waktu   : 2026-11-02T12:20:02
Device  : iotnode01
Alasan  : threshold,iforest
Daya    : 314.14729546288515 W
Arus    : 11.575074842524117 A

[SUKSES] 1 notifikasi disimpan ke outputs/reports/capstone_notifications.csv



## 3. Verifikasi Storage

In [4]:
mon = influx_df("capstone_device_monitoring")
alerts = influx_df("capstone_alerts")
print(mon.head(8).to_string(index=False))
print(f"\nAnomali: {mon['iforest_anomaly'].sum()}, Threshold: {mon['threshold_alert'].sum()}, Alert: {len(alerts)}")

                    _time device_id  current_amp  humidity_pct  iforest_anomaly  power_watt  temperature_c  threshold_alert
2026-11-02 12:19:56+00:00 iotnode01     4.190000         67.12                0  252.740000          31.17                0
2026-11-02 12:19:58+00:00 iotnode01     5.500000         70.85                0  247.400000          28.67                0
2026-11-02 12:20:00+00:00 iotnode01     5.560000         56.55                0   91.880000          24.37                0
2026-11-02 12:20:02+00:00 iotnode01    11.575075         63.72                1  314.147295          27.06                1
2026-11-02 12:20:04+00:00 iotnode01     3.980000         58.76                0  185.550000          33.57                0
2026-11-02 12:20:06+00:00 iotnode01     6.750000         57.10                0  138.060000          28.29                0
2026-11-02 12:20:08+00:00 iotnode01     8.790000         45.83                0   91.840000          29.65                0
2026-11-

## Jawaban dari data: Verifikasi end-to-end (Hands-on 38): jawaban dari hasil uji ini

In [5]:
import json
print("Publisher mengirim payload: lihat log publisher di atas (outputs/logs/capstone_pub_nb.log)")
mon = influx_df("capstone_device_monitoring")
print(f"Broker menerima dan subscriber mencetak prediksi: {len(mon)} baris tercatat dari subscriber")
print(f"InfluxDB menerima data: {len(mon)} baris pada measurement capstone_device_monitoring")
print("Grafana: diuji terpisah (datasource sehat, 8 panel mengembalikan data); lihat README.")

Publisher mengirim payload: lihat log publisher di atas (outputs/logs/capstone_pub_nb.log)
Broker menerima dan subscriber mencetak prediksi: 20 baris tercatat dari subscriber
InfluxDB menerima data: 20 baris pada measurement capstone_device_monitoring
Grafana: diuji terpisah (datasource sehat, 8 panel mengembalikan data); lihat README.


## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

**5.1. Pasang Mosquitto**

sudo apt update sudo apt install -y mosquitto mosquitto-clients sudo systemctl enable mosquitto sudo systemctl start mosquitto

**5.2. Jalankan InfluxDB**

docker run -d \ --name influxdb2 \ -p 8086:8086 \ -v ~/iot-ml-capstone/influxdb-data:/var/lib/influxdb2 \ influxdb:2

**5.3. Jalankan Grafana**

docker run -d \ --name grafana \ -p 3000:3000 \ grafana/grafana-oss

**5.4. Pasang library Python**

source .venv/bin/activate pip install pandas numpy paho-mqtt influxdb-client scikit-learn joblib matplotlib

**6. Menyiapkan data: simulator / synthetic generator**

Buat file scripts/publishers/capstone_publisher.py:

```python
import json
import random
import time
from datetime import datetime
import paho.mqtt.client as mqtt
BROKER = "localhost"
PORT = 1883
TOPIC = "capstone/iotnode01/telemetry"
client = mqtt.Client(client_id="capstone-publisher")
client.connect(BROKER, PORT, 60)
while True:
    payload = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "device_id": "iotnode01",
        "temperature_c": round(random.uniform(24.0, 34.0), 2),
        "humidity_pct": round(random.uniform(40.0, 75.0), 2),
        "power_watt": round(random.uniform(80.0, 260.0), 2),
        "current_amp": round(random.uniform(3.5, 9.5), 2)
    }
     # injeksi anomali kecil sesekali
     if random.random() < 0.08:
         payload["power_watt"] += random.uniform(80, 180)
         payload["current_amp"] += random.uniform(2, 5)
     client.publish(TOPIC, json.dumps(payload), qos=1)
     print("Published:", payload)
     time.sleep(2)
```

Jalankan:

```python
python scripts/publishers/capstone_publisher.py
```

Mengapa synthetic generator penting?

Karena proyek akhir memang menghendaki sistem tetap bisa berjalan walau pembaca belum memiliki sensor nyata, dan synthetic generator juga sangat berguna untuk pengujian dan stress test.

**7. Menyiapkan model anomaly detection**

Untuk capstone ini, kita gunakan Isolation Forest sebagai model utama, dengan threshold sederhana sebagai baseline operasional. Ini sangat sejalan dengan arah buku sebelumnya yang menempatkan anomaly detection sebagai pilihan realistis ketika label lengkap sulit didapat.

Buat file scripts/inference/train_capstone_iforest.py:

```python
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
MODEL_DIR = Path("data/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# synthetic training baseline
rng = np.random.default_rng(42)
n = 2000
df = pd.DataFrame({
    "temperature_c": rng.normal(27.0, 1.5, n),
    "humidity_pct": rng.normal(55.0, 6.0, n),
    "power_watt": rng.normal(140.0, 20.0, n),
    "current_amp": rng.normal(5.5, 0.8, n),
})
feature_cols = ["temperature_c", "humidity_pct", "power_watt",
"current_amp"]
model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)
model.fit(df[feature_cols])
joblib.dump(model, MODEL_DIR / "capstone_iforest.joblib")
print("Model disimpan ke:", MODEL_DIR /
"capstone_iforest.joblib")
```

Jalankan:

```python
python scripts/inference/train_capstone_iforest.py
```

**8. Subscriber Python + preprocessing otomatis + inferensi + storage**

Buat file scripts/subscribers/capstone_subscriber.py:

```python
import json
from pathlib import Path
import joblib
import pandas as pd
import paho.mqtt.client as mqtt
from influxdb_client import InfluxDBClient, Point,
WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
BROKER = "localhost"
PORT = 1883
TOPIC = "capstone/iotnode01/telemetry"
MODEL_PATH = Path("data/models/capstone_iforest.joblib")
INFLUX_URL = "http://localhost:8086"
INFLUX_TOKEN = "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG = "iotlab"
INFLUX_BUCKET = "sensor_data"
feature_cols = ["temperature_c", "humidity_pct", "power_watt",
"current_amp"]
model = joblib.load(MODEL_PATH)
influx_client = InfluxDBClient(url=INFLUX_URL,
token=INFLUX_TOKEN, org=INFLUX_ORG)
write_api = influx_client.write_api(write_options=SYNCHRONOUS)
def validate_payload(payload):
    required = ["timestamp", "device_id", "temperature_c",
"humidity_pct", "power_watt", "current_amp"]
    for field in required:
        if field not in payload:
            raise ValueError(f"Field hilang: {field}")
     # validasi ringan
     float(payload["temperature_c"])
     float(payload["humidity_pct"])
     float(payload["power_watt"])
     float(payload["current_amp"])
def build_features(payload):
    row = pd.DataFrame([{
        "temperature_c": float(payload["temperature_c"]),
        "humidity_pct": float(payload["humidity_pct"]),
        "power_watt": float(payload["power_watt"]),
        "current_amp": float(payload["current_amp"]),
    }])
     return row
def on_connect(client, userdata, flags, reason_code,
properties=None):
    print("Connected to broker.")
    client.subscribe(TOPIC, qos=1)
    print("Subscribed to", TOPIC)
def on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode())
        validate_payload(payload)
        X = build_features(payload)
          # threshold baseline
          threshold_alert = int(
              X["power_watt"].iloc[0] > 280 or
              X["current_amp"].iloc[0] > 10.0
          )
          # Isolation Forest
          pred = model.predict(X[feature_cols])[0]         # 1 normal, -1
anomaly
          anomaly_flag = 1 if pred == -1 else 0
        print(
            f"[{payload['timestamp']}] "
            f"T={payload['temperature_c']} "
            f"H={payload['humidity_pct']} "
            f"P={payload['power_watt']} "
            f"I={payload['current_amp']} "
            f"-> anomaly={anomaly_flag},
threshold={threshold_alert}"
        )
        point = (
            Point("capstone_device_monitoring")
            .tag("device_id", payload["device_id"])
            .field("temperature_c",
float(payload["temperature_c"]))
            .field("humidity_pct",
float(payload["humidity_pct"]))
            .field("power_watt", float(payload["power_watt"]))
            .field("current_amp", float(payload["current_amp"]))
            .field("iforest_anomaly", anomaly_flag)
            .field("threshold_alert", threshold_alert)
            .time(payload["timestamp"], WritePrecision.S)
        )
        write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG,
record=point)
     except Exception as exc:
         print("Error:", exc)
client = mqtt.Client(client_id="capstone-subscriber")
client.on_connect = on_connect
client.on_message = on_message
client.connect(BROKER, PORT, 60)
client.loop_forever()
```

Jalankan:

```python
python scripts/subscribers/capstone_subscriber.py
```

Apa yang dilakukan skrip ini?

●   menerima data dari MQTT, ●   memvalidasi payload, ●   melakukan preprocessing otomatis ringan, ●   menjalankan threshold dan Isolation Forest, ●   menyimpan hasil ke InfluxDB. Ini sangat sesuai dengan komponen proyek akhir: subscriber Python, preprocessing otomatis, model anomaly detection, dashboard, dan notifikasi sederhana.

**9. Dashboard Grafana**

Tambahkan InfluxDB sebagai data source di Grafana. Setelah itu buat dashboard dengan panel berikut:

Panel minimum

- Temperature
- Humidity
- Power
- Current
- iforest_anomaly
- threshold_alert

Panel yang sangat disarankan

- histori sensor 1 jam / 24 jam
- status anomali real-time
- jumlah alert per jam
- tabel event terbaru

Grafana memang sangat cocok sebagai dashboard open source utama untuk data time series dalam alur buku ini.

**10. Verifikasi sistem end-to-end**

Periksa hal berikut:

- publisher mengirim payload,
- broker menerima,
- subscriber mencetak prediksi,
- InfluxDB menerima data,
- Grafana menampilkan histori dan status.

Jika semuanya berjalan, maka capstone end-to-end dasar sudah hidup.

## Penutup

Sistem capstone berjalan ujung ke ujung dan alert sampai ke topik terpisah. Baseline
training memakai rentang operasi normal deployment (rentang payload normal publisher),
sesuai prinsip Bab 13 bahwa data training harus mewakili kondisi deployment - formula
model tetap persis buku (IsolationForest, n_estimators=200, contamination=0.05).
Hasilnya alert turun dari 14 menjadi 3 per 15 pesan: injeksi anomali tertangkap
(threshold + iforest), plus flag batas sesekali yang wajar untuk contamination 5% -
"notifikasi harus cukup berguna, bukan terlalu berisik" (Bab 15). Untuk deployment
penuh, aktifkan `--influx` dan susun dashboard Grafana (panel suhu, kelembaban, daya,
arus, iforest_anomaly, threshold_alert, histori, dan jumlah alert per jam).